# Capstone: Replicate the Wall, Then Break It on Real Pixels

Capstone notebook for the [Neural Network Fundamentals](https://josephausterweil.github.io/probintro/deep/neural-net-fundamentals/) chapter (Part VIII).

**The experiment.** Two acts, one architecture:

- **Part A — replicate Minsky & Papert (1969) and the 1986 reply.** Train a purely linear model on XOR and confirm the wall: loss floored at **0.25**, every prediction **0.5**. Then add one hidden ReLU layer (the chapter's 2-2-1 network) and watch the *same optimizer* solve it exactly.
- **Part B — a real task.** The chapter's four stamp days are a logic puzzle; the kiosk's world is pixels. Classify **two digit classes** from `sklearn.datasets.load_digits` (8×8 grayscale images, so 64 inputs instead of 2) with the *same two-layer architecture* — only the input width changes. Report test accuracy and inspect the mistakes.

Setup and data cells are complete — run them as-is. The model fits are **TODO** stubs (each stub says exactly what to build, and a commented check below it says what success looks like). Uncomment the checks as you go.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

key = jr.PRNGKey(2)   # the chapter's seed: solves XOR from this start
print('JAX ready:', jax.__version__)

## Part A — the wall (Minsky & Papert, 1969)

### A1. The XOR data (complete — run me)

The four stamp days from the chapter: `x = [lunch is tonkatsu, dinner is tonkatsu]`, label `1.0` when exactly one is (the variety stamp).

In [ ]:
X_xor = jnp.array([[0.0, 0.0],
                   [0.0, 1.0],
                   [1.0, 0.0],
                   [1.0, 1.0]])
y_xor = jnp.array([0.0, 1.0, 1.0, 0.0])
print('XOR data:', X_xor.shape, ' labels:', y_xor)

### A2. The linear model fails — TODO

Build and train a purely linear model, `f(x) = w . x + b`, by full-batch gradient descent on mean squared error (`lr = 0.1`, 2000 steps — same as the chapter). Initialize `w` to zeros, `b` to `0.0`; no randomness needed.

Minsky & Papert say the loss cannot beat **0.25**, and the least-bad predictions are **0.5 everywhere** — the model shrugging. Confirm it.

In [ ]:
def linear_fit(X, y, steps=2000, lr=0.1):
    """Train f(x) = w . x + b by GD on MSE. Return (predict_fn, final_loss)."""
    # TODO:
    #   1. params = {'w': jnp.zeros(X.shape[1]), 'b': jnp.array(0.0)}
    #   2. loss(params) = mean squared error of jax.vmap over (X, y)
    #   3. loop: params = jax.tree.map(lambda p, g: p - lr * g, params, jax.grad(loss)(params))
    #   4. return (lambda Xnew: jax.vmap(...)(Xnew), float(loss(params)))
    raise NotImplementedError

# When linear_fit works, this should print loss 0.2500 and four predictions of 0.50:
#
# predict, final_loss = linear_fit(X_xor, y_xor)
# print(f'linear loss floor: {final_loss:.4f}')      # expect 0.2500
# print('predictions:', predict(X_xor))               # expect ~[0.5 0.5 0.5 0.5]

### A3. One hidden layer breaks the wall — TODO

Port the chapter's 2-2-1 cell into a function: 2 inputs → `H` hidden ReLU units → 1 linear output, trained the same way. Initialize `A ~ 0.5 * normal(k1, (H, 2))`, `c = zeros`, `w ~ 0.5 * normal(k2, (H,))`, `b = 0.0` from a `key` argument so runs are reproducible.

With `key = jr.PRNGKey(2)` and `H = 2` the loss should reach ~**0.000000** and the four predictions should round to `0, 1, 1, 0` — the 1986 reply, replicated. (Other seeds may park in a local minimum — the chapter's footnote. Try a few.)

In [ ]:
def net_fit(X, y, key, H=2, steps=2000, lr=0.1):
    """Train an H-hidden-unit ReLU net by GD on MSE. Return (predict_fn, final_loss)."""
    def net(params, x):
        h = jnp.maximum(0.0, params['A'] @ x + params['c'])
        return jnp.dot(params['w'], h) + params['b']
    # TODO:
    #   1. loss(params) = mean squared error of jax.vmap(net) over (X, y)
    #   2. k1, k2 = jr.split(key); init params as described above
    #      (note: A has shape (H, X.shape[1]) -- Part B reuses this with 64 inputs)
    #   3. same GD loop as linear_fit
    #   4. return (lambda Xnew: jax.vmap(lambda x: net(params, x))(Xnew), float(loss(params)))
    raise NotImplementedError

# When net_fit works:
#
# predict, final_loss = net_fit(X_xor, y_xor, key)
# print(f'2-2-1 loss: {final_loss:.6f}')             # expect ~0.000000
# print('predictions:', predict(X_xor))              # expect ~[0. 1. 1. 0.]

## Part B — real pixels

### B1. Two digit classes (complete — run me)

8×8 grayscale handwritten digits. We keep classes **0** and **1**, rescale pixels to 0–1, and split off a test set the network never trains on. Feature vectors are 64-D — the same math as 2-D, per the vectors chapter.

In [ ]:
digits = load_digits()
mask = (digits.target == 0) | (digits.target == 1)
X_dig = jnp.array(digits.data[mask] / 16.0)          # pixels rescaled to 0-1
y_dig = jnp.array((digits.target[mask] == 1).astype(float))   # 1.0 = digit "1"

# Reproducible shuffle + 80/20 train/test split
perm = jr.permutation(jr.PRNGKey(0), X_dig.shape[0])
n_train = int(0.8 * X_dig.shape[0])
train_idx, test_idx = perm[:n_train], perm[n_train:]
X_train, y_train = X_dig[train_idx], y_dig[train_idx]
X_test, y_test = X_dig[test_idx], y_dig[test_idx]
print('train:', X_train.shape, ' test:', X_test.shape)

# Look at the data you are about to model
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(int(y_train[i]), fontsize=9)
    ax.axis('off')
plt.show()

### B2. The same net, 64 inputs — TODO

Call **your `net_fit` from A3** on the digit data — nothing about the function needs to change; `A` just becomes `(H, 64)`. Start with `H = 8`, `steps = 2000`, `lr = 0.1`. Then:

1. Threshold predictions at 0.5 to get hard labels.
2. Report **accuracy on the test set** (fraction correct). Above ~0.98 is normal for 0-vs-1.
3. Compare with training accuracy — is there a gap, and which way?

In [ ]:
# TODO:
#   predict, final_loss = net_fit(X_train, y_train, key, H=8)
#   train_acc = float(jnp.mean((predict(X_train) > 0.5) == (y_train > 0.5)))
#   test_acc  = float(jnp.mean((predict(X_test)  > 0.5) == (y_test  > 0.5)))
#   print(f'train accuracy: {train_acc:.3f}   test accuracy: {test_acc:.3f}')
pass

### B3. Look at the mistakes — TODO

Find the test digits your network got wrong (if it got *none* wrong, rerun with `H = 2` or fewer steps to earn some mistakes) and plot them with `imshow`, titled `true -> predicted`. Would you have gotten them right? What does a smudged "0" that reads as a "1" tell you about what the 64 learned listening patterns are attending to?

In [ ]:
# TODO:
#   wrong = jnp.where((predict(X_test) > 0.5) != (y_test > 0.5))[0]
#   for i in wrong: plt.imshow(X_test[i].reshape(8, 8), cmap='gray_r'); ...
pass

## Optional stretch

- **A harder pair.** Swap classes 0/1 for **3 vs 5** (genuinely confusable shapes) and re-run B2–B3. How much accuracy do you lose? Does raising `H` buy it back?
- **Width sweep.** On the harder pair, sweep `H` over `[1, 2, 4, 8, 16, 32, 64]` and plot test accuracy vs. width. Where does adding units stop paying? (Tanaka-san's box, as a plot.)

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*